# oruk-loop-153M — looped English ASR on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/oruk-loop-153M-looped-asr.ipynb)

[`oruk/oruk-loop-153M`](https://huggingface.co/oruk/oruk-loop-153M) is Oruk AI's English automatic speech recognizer (**153,202,638 stored parameters**, released 26 October 2026). It is distilled from [`Qwen/Qwen3-ASR-1.7B-hf`](https://huggingface.co/Qwen/Qwen3-ASR-1.7B-hf) (2,038,052,480 stored parameters, 13.3× larger) and keeps that teacher's architecture — except each transformer stack is **one shared block run in a loop**:

- one audio-encoder block applied **24** times
- one text-decoder block applied **28** times
- **low-rank per-depth adapters** (rank 64 on the encoder, rank 16 on the decoder) tell each pass which layer it is standing in for

Every token still walks 24 + 28 layers at the teacher's width. Looping shrinks **storage**, not arithmetic. The card's headline number is a **mean WER of 7.12%** with beam-4 decoding (7.40% greedy) on the seven English [Open ASR Leaderboard](https://huggingface.co/spaces/hf-audio/open-asr-leaderboard) sets, from Oruk's harness rather than an official submission.

This notebook is the **looped-transformer / parameter-sharing** teaching counterpart of the existing speech notebooks. The [AIBackends Whisper notebook](https://github.com/donvito/notebooks/blob/main/colab/AIBackends-Whisper-audio-transcription.ipynb) is faster-whisper ASR plus PII redaction. The [Granite Speech notebook](https://github.com/donvito/notebooks/blob/main/colab/granite-speech-5.0-2b-asr-translation.ipynb) is promptable multilingual ASR, keyword bias, and X↔En translation. This one is English-only, loads with `trust_remote_code`, and shows *why 153M stored weights can still run a 1.7B-shaped stack*.

## Licence — weights are not standard open source

Read this before you run anything.

| Files | Licence |
| --- | --- |
| `model.safetensors` (weights) | **[PolyForm Small Business 1.0.0](https://polyformproject.org/licenses/small-business/1.0.0) or [PolyForm Noncommercial 1.0.0](https://polyformproject.org/licenses/noncommercial/1.0.0)**, at your option. This is **not** MIT, Apache, or another OSI licence. Other commercial use needs a [license from Oruk AI](https://oruk.ai). |
| `modeling_oruk_loop.py` (custom inference code) | [MIT](https://huggingface.co/oruk/oruk-loop-153M/blob/main/LICENSE) |
| Unmodified Qwen3-ASR tokenizer / chat template / processor | [Apache-2.0](https://huggingface.co/oruk/oruk-loop-153M/blob/main/LICENSE-QWEN) |

**Required Notice:** Copyright (c) 2026 Oruk AI (https://oruk.ai)

Keep that notice and the [NOTICE.md](https://huggingface.co/oruk/oruk-loop-153M/blob/main/NOTICE.md) attribution (Qwen3-ASR-1.7B, © Alibaba Cloud / the Qwen team, Apache-2.0) with any copy of the weights you distribute. Small businesses (fewer than 100 employees and contractors, under US$1M prior-year revenue) may use the weights commercially under PolyForm Small Business. Anyone may use them for personal, research, and educational purposes under PolyForm Noncommercial. Charities, schools, universities, public research organisations, and government bodies may use them freely. **Any other commercial use needs a commercial license from Oruk AI.**

Some training corpora (SPGISpeech, GigaSpeech, TED-LIUM 3) carry non-commercial or research-only terms on the *data*. Review those terms for your own use. No endorsement by Qwen, Alibaba Cloud, or the dataset providers is implied.

`library_name: transformers`. The Hub repo ships custom code, so **`trust_remote_code=True`** is required. Pin a revision if you need a frozen snapshot.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has ~16 GB of VRAM and **no native bfloat16**. The card's CUDA default is bf16 autocast plus merged per-depth weights (~4.2 GB after load). This notebook loads **float16** on T4 (`dtype=torch.float16`, `merge=True`). If transcripts look degraded, set `DTYPE = torch.float32` in the GPU cell and reload. CPU works in float32 (~3.9 GB RAM) but is slower than real time; this notebook expects a T4.

Stored weights are **~583 MB** (`model.safetensors`). Merged decode materialises a full-depth copy of each linear, so peak VRAM is several GB, not 0.6 GB.

**Links:** [model card](https://huggingface.co/oruk/oruk-loop-153M) · [weights licence](https://huggingface.co/oruk/oruk-loop-153M/blob/main/LICENSE-WEIGHTS.md) · [NOTICE](https://huggingface.co/oruk/oruk-loop-153M/blob/main/NOTICE.md) · [teacher](https://huggingface.co/Qwen/Qwen3-ASR-1.7B-hf) · [Open ASR Leaderboard](https://huggingface.co/spaces/hf-audio/open-asr-leaderboard) · [oruk.ai](https://oruk.ai)


In [ ]:
%pip install -q "transformers>=5.17.0" datasets soundfile soxr jiwer librosa

# Card install is: pip install "transformers==5.17.0" torch soundfile soxr
# Do NOT pip install torch here. Colab already ships a CUDA wheel; forcing a
# reinstall often drops GPU support or pulls a CPU-only build.
# transformers>=5.17.0 is the floor (Qwen3ASRForConditionalGeneration).
# datasets + librosa/soundfile load the tiny LibriSpeech clip set.
# soxr is the card's resampler; jiwer scores WER in demo 2.


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The upgrade is invisible until the process restarts.


## 1. Check the GPU

Confirm CUDA is available, print the device and free VRAM, and pick a dtype. T4 (`sm_75`) has no native bfloat16 — **float16** is the path this notebook uses. Fall back to float32 if outputs degrade.


In [ ]:
import os
import time

import torch
import transformers
from packaging.version import Version

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

assert Version(transformers.__version__) >= Version("5.17.0"), (
    f"transformers {transformers.__version__} is loaded; oruk-loop-153M needs 5.17.0+ "
    "(Qwen3ASRForConditionalGeneration). Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

major, minor = torch.cuda.get_device_capability()
DEVICE = "cuda"
# T4 (sm_75) has no native bfloat16. The card's CUDA default is bf16 autocast.
# float16 is the T4 path. Set DTYPE = torch.float32 if transcripts look degraded.
DTYPE = torch.float16
MODEL_ID = "oruk/oruk-loop-153M"
TARGET_SR = 16_000

free_b, total_b = torch.cuda.mem_get_info()
props = torch.cuda.get_device_properties(0)
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
print(f"dtype: {DTYPE}  (card default on CUDA is bf16; T4 has no native bf16)")
print(f"VRAM total: {total_b / 1024**3:.2f} GB | free: {free_b / 1024**3:.2f} GB")
print(f"device capability {major}.{minor} | multiprocessors {props.multi_processor_count}")
if major < 8:
    print("No native bfloat16 on this GPU — staying on float16.")
else:
    print("This GPU can run bfloat16; this notebook still uses float16 so a T4 recipe stays consistent.")


def print_memory(prefix=""):
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    free_now, total_now = torch.cuda.mem_get_info()
    print(
        f"{prefix}VRAM allocated: {allocated:.2f} GB | reserved: {reserved:.2f} GB | "
        f"peak: {peak:.2f} GB | free: {free_now / 1024**3:.2f}/{total_now / 1024**3:.2f} GB"
    )


def timed(label, fn):
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    result = fn()
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    print(f"{label}: {elapsed:.2f}s")
    print_memory(prefix=f"{label}  ")
    return result, elapsed


## 2. Load oruk-loop-153M

`AutoModel.from_pretrained(..., trust_remote_code=True)` runs [`modeling_oruk_loop.py`](https://huggingface.co/oruk/oruk-loop-153M/blob/main/modeling_oruk_loop.py) (MIT) from the Hub repo. That file subclasses stock `Qwen3ASRForConditionalGeneration`. `AutoProcessor` needs **no** remote code — tokenizer, chat template, and feature extractor are the unmodified Qwen3-ASR files (Apache-2.0).

The first run downloads about **583 MB**. `merge=True` materialises `W + B_d A_d` for every depth so decode is a single matmul per layer instead of shared-weight + adapter. That is the card's CUDA default and the reason load VRAM is a few GB, not 0.6 GB.


In [ ]:
from transformers import AutoModel, AutoProcessor

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

t0 = time.perf_counter()
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    device=DEVICE,
    dtype=DTYPE,
    merge=True,
)
torch.cuda.synchronize()
load_s = time.perf_counter() - t0

stored = model.num_stored_parameters()
loop = dict(getattr(model.config, "oruk_loop", {}) or {})
print(f"loaded {MODEL_ID} in {load_s:.1f}s")
print("architecture:", model.config.architectures)
print("dtype:", next(model.parameters()).dtype, "| device:", model.device)
print(f"stored parameters: {stored:,}  (card oruk_loop.stored_parameters={loop.get('stored_parameters')})")
print_memory("after load  ")
print()
print("If greedy transcripts later look empty or garbled, reload with DTYPE = torch.float32.")


## 3. Demo 1 — one public-domain clip, greedy vs beam 4

A short LibriSpeech *clean* excerpt from [`hf-internal-testing/librispeech_asr_dummy`](https://huggingface.co/datasets/hf-internal-testing/librispeech_asr_dummy) (73 clips, ~9 MB). These are public-domain LibriVox audiobook readings, the same family as OpenSLR LibriSpeech (CC BY 4.0). This is **not** the official `test-clean` leaderboard split — it is a stable, tiny set so the notebook does not pull the full corpus.

`model.transcribe` is the card API. `num_beams=1` is greedy; `num_beams=4` is the setting behind the 7.12% mean. Output keeps the teacher's casing and punctuation.


In [ ]:
from datasets import Audio, load_dataset
from IPython.display import Audio as play, display

ds = load_dataset(
    "hf-internal-testing/librispeech_asr_dummy",
    "clean",
    split="validation",
)
ds = ds.cast_column("audio", Audio(sampling_rate=TARGET_SR))

clip = ds[0]
wav = clip["audio"]["array"]
sr = int(clip["audio"]["sampling_rate"])
seconds = len(wav) / sr
ref = clip["text"]

print(f"id: {clip['id']}  |  {seconds:.2f}s  |  sr={sr}")
print("reference:", ref)
display(play(wav, rate=sr))

print("\n=== greedy (num_beams=1) ===")
greedy, greedy_s = timed(
    "greedy",
    lambda: model.transcribe((wav, sr), processor, num_beams=1),
)
print(greedy)

print("\n=== beam 4 (headline setting) ===")
beamed, beam_s = timed(
    "beam-4",
    lambda: model.transcribe((wav, sr), processor, num_beams=4),
)
print(beamed)

print(f"\nreal-time factor  greedy: {greedy_s / seconds:.2f}x   beam-4: {beam_s / seconds:.2f}x")
print("(RTF < 1 means faster than real time.)")


## 4. Demo 2 — WER on a handful of LibriSpeech clean clips

Five short clips from the same dummy set, scored with [`jiwer`](https://jitsi.github.io/jiwer/). The Open ASR Leaderboard applies Whisper's `EnglishTextNormalizer` before WER; we do the same when transformers still ships it, otherwise a small lowercase / punctuation fallback. This is a **smoke check**, not a leaderboard number (the card's LibriSpeech test-clean beam-4 WER is 2.40%).


In [ ]:
import re

import jiwer

try:
    from transformers.models.whisper.english_normalizer import EnglishTextNormalizer

    _normalize = EnglishTextNormalizer()
    NORM_NAME = "whisper EnglishTextNormalizer"
except Exception:
    def _normalize(text):
        text = text.lower()
        text = re.sub(r"[^a-z0-9'\s]", " ", text)
        return " ".join(text.split())

    NORM_NAME = "lowercase + strip punctuation (fallback)"


def pick_short(dataset, n=5, max_seconds=12.0):
    chosen = []
    for row in dataset:
        audio = row["audio"]
        dur = len(audio["array"]) / audio["sampling_rate"]
        if dur <= max_seconds:
            chosen.append(row)
        if len(chosen) >= n:
            break
    return chosen


samples = pick_short(ds, n=5, max_seconds=12.0)
print(f"scoring {len(samples)} clips with {NORM_NAME}")
for row in samples:
    dur = len(row["audio"]["array"]) / row["audio"]["sampling_rate"]
    print(f"  {row['id']}: {dur:.2f}s")

batch = [(row["audio"]["array"], int(row["audio"]["sampling_rate"])) for row in samples]
refs = [row["text"] for row in samples]
audio_s = sum(len(wav) / sr for wav, sr in batch)

hyps, wer_s = timed(
    "beam-4 batch",
    lambda: model.transcribe(batch, processor, num_beams=4, batch_size=5),
)

print(f"\naudio {audio_s:.1f}s  |  RTF {wer_s / audio_s:.2f}x")
print(f"{'id':<20} {'wer':>6}  hypothesis")
print("-" * 72)
wers = []
for row, ref, hyp in zip(samples, refs, hyps):
    nref, nhyp = _normalize(ref), _normalize(hyp)
    w = jiwer.wer(nref, nhyp)
    wers.append(w)
    print(f"{row['id']:<20} {100 * w:5.1f}%  {hyp}")
    print(f"{'':<20} {'ref:':>6}  {ref}")

micro = jiwer.wer([_normalize(r) for r in refs], [_normalize(h) for h in hyps])
print(f"\nmicro WER (edits / ref words): {100 * micro:.2f}%")
print(f"macro WER (mean of per-clip):  {100 * (sum(wers) / len(wers)):.2f}%")
print("Card LibriSpeech test-clean beam-4 WER is 2.40% — this handful is not that set.")


## 5. Demo 3 — light faster-whisper comparison

A side-by-side on the same five clips against [`faster-whisper`](https://github.com/SYSTRAN/faster-whisper) `tiny.en` (the engine behind the [AIBackends Whisper notebook](https://github.com/donvito/notebooks/blob/main/colab/AIBackends-Whisper-audio-transcription.ipynb)). This is a **speed/size** contrast, not a quality peer: tiny.en is ~39M and English-only; oruk-loop-153M is a 153M stored student of a 1.7B teacher.

The Qwen3-ASR-1.7B teacher is skipped. Its stored weights are ~4 GB in float16, and the card notes that *merged* oruk-loop decode already needs about as much accelerator memory as the teacher. Loading the teacher on a free T4 next to this student is not a light demo.

`faster-whisper` is CTranslate2 — it does not replace Colab's torch.


In [ ]:
%pip install -q faster-whisper

# CTranslate2 runtime. Does not reinstall torch.


In [ ]:
from faster_whisper import WhisperModel

fw = WhisperModel("tiny.en", device="cuda", compute_type="float16")
print("faster-whisper tiny.en ready")
print_memory("after tiny.en load  ")


def whisper_one(wav, sr):
    segments, _info = fw.transcribe(wav, language="en", beam_size=1, vad_filter=False)
    return " ".join(seg.text.strip() for seg in segments).strip()


def whisper_batch():
    return [whisper_one(wav, sr) for wav, sr in batch]


fw_hyps, fw_s = timed("faster-whisper tiny.en", whisper_batch)
fw_micro = jiwer.wer([_normalize(r) for r in refs], [_normalize(h) for h in fw_hyps])
oruk_micro = micro

print()
print(f"{'model':<28} {'params':>10} {'time':>8} {'RTF':>6} {'micro WER':>10}")
print("-" * 68)
print(
    f"{'oruk-loop-153M beam-4':<28} {'153M stored':>10} {wer_s:7.2f}s "
    f"{wer_s / audio_s:5.2f}x {100 * oruk_micro:9.2f}%"
)
print(
    f"{'faster-whisper tiny.en':<28} {'~39M':>10} {fw_s:7.2f}s "
    f"{fw_s / audio_s:5.2f}x {100 * fw_micro:9.2f}%"
)
print()
for row, o, w in zip(samples, hyps, fw_hyps):
    print(f"{row['id']}")
    print(f"  oruk:    {o}")
    print(f"  whisper: {w}")

del fw
torch.cuda.empty_cache()
print()
print_memory("after releasing tiny.en  ")


## 6. Demo 4 — stored vs effective layers

The config still *reports* a 24-layer encoder and a 28-layer decoder (`audio_config.encoder_layers`, `text_config.num_hidden_layers`) because that is the compute graph. What is **stored** is one shared block per stack plus per-depth adapters. `oruk_loop` on the config is the teaching table.


In [ ]:
from collections import Counter

raw = getattr(model.config, "oruk_loop", None)
if isinstance(raw, dict):
    loop = dict(raw)
elif raw is None:
    loop = {}
elif hasattr(raw, "keys"):
    loop = dict(raw)
else:
    loop = {k: getattr(raw, k) for k in dir(raw) if not k.startswith("_")}

# Config still reports the teacher depths even if oruk_loop was dropped by AutoConfig.
loop.setdefault("encoder_loops", model.config.audio_config.encoder_layers)
loop.setdefault("decoder_loops", model.config.text_config.num_hidden_layers)
loop.setdefault("encoder_rank", 64)
loop.setdefault("decoder_rank", 16)
loop.setdefault("table_rows", 36974)
loop.setdefault("table_exact", 368)
loop.setdefault("table_rank", 729)
loop.setdefault("stored_parameters", model.num_stored_parameters())

print("config.oruk_loop (plus card fallbacks if a key is missing)")
for key in (
    "encoder_loops",
    "decoder_loops",
    "encoder_rank",
    "decoder_rank",
    "table_rows",
    "table_exact",
    "table_rank",
    "stored_parameters",
):
    print(f"  {key}: {loop[key]}")

print()
print("Teacher-shaped depths still in the Qwen3-ASR configs")
print(f"  audio_config.encoder_layers     = {model.config.audio_config.encoder_layers}")
print(f"  text_config.num_hidden_layers   = {model.config.text_config.num_hidden_layers}")
print(f"  text_config.num_attention_heads = {model.config.text_config.num_attention_heads}")
print(f"  text_config.num_key_value_heads = {model.config.text_config.num_key_value_heads}")

enc_layers = model.model.audio_tower.layers
dec_layers = model.model.language_model.layers
print()
print("What is actually constructed")
print(f"  audio_tower.layers      : {len(enc_layers)} × {type(enc_layers[0]).__name__}  (parameter-free call sites)")
print(f"  audio_tower.loop        : {type(model.model.audio_tower.loop).__name__}  (the ONE shared encoder block)")
print(f"  language_model.layers   : {len(dec_layers)} × {type(dec_layers[0]).__name__}")
print(f"  language_model.loop     : {type(model.model.language_model.loop).__name__}  (the ONE shared decoder block)")

print()
print("Stored vs effective")
print(f"  encoder stored blocks : 1     effective layers: {loop['encoder_loops']}")
print(f"  decoder stored blocks : 1     effective layers: {loop['decoder_loops']}")
print(f"  stored parameters     : {model.num_stored_parameters():,}")
print("  teacher stored params : 2,038,052,480  (13.3×)")
print()
print("Each looped linear is W x + b_d + B_d (A_d x): W is shared; A_d, B_d, b_d belong to depth d.")
print("Shared weights ~62.9M; per-depth adapters ~45.7M; kept English token table 36,974 rows.")

kinds = Counter(type(m).__name__ for m in model.modules())
print()
print("module census (interesting types):")
for name in ("LoopLinear", "LoopLayerNorm", "LoopRMSNorm", "LoopCall", "TokenTable"):
    if name in kinds:
        print(f"  {name}: {kinds[name]}")

print()
print_memory("after architecture inspect  ")


## 7. Cleanup

Release the student so later cells (or a second model) get the VRAM back. Then read the caveats — especially the weights licence — before you take this past a teaching session.


In [ ]:
import gc

print_memory("before cleanup  ")
try:
    del model
except NameError:
    pass
try:
    del processor
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print_memory("after cleanup   ")
print("Student released. Re-run the load cell if you want another clip.")


## Caveats

**Weights licence.** The checkpoint is **not** standard open source. PolyForm Small Business 1.0.0 **or** PolyForm Noncommercial 1.0.0, at your option. Other commercial use needs a license from Oruk AI. Keep **Required Notice: Copyright (c) 2026 Oruk AI (https://oruk.ai)** and the [NOTICE.md](https://huggingface.co/oruk/oruk-loop-153M/blob/main/NOTICE.md) Qwen3-ASR attribution with any copy of the weights.

**`trust_remote_code`.** `modeling_oruk_loop.py` executes locally (MIT). Review it, and pin a revision for reproduction. The card was tested with `transformers==5.17.0`.

**English only.** The vocabulary is trimmed to 36,974 English / special / prompt tokens. The prompt is `language="English"`. There are no timestamps and no diarization. Pair this notebook with [Nemotron-3 Diarization](Nemotron-3-Diarization.ipynb) if you need who-spoke-when, or with [Granite Speech](granite-speech-5.0-2b-asr-translation.ipynb) if you need translation or keyword bias.

**Not the Whisper notebook.** [`AIBackends-Whisper-audio-transcription.ipynb`](https://github.com/donvito/notebooks/blob/main/colab/AIBackends-Whisper-audio-transcription.ipynb) is local faster-whisper plus PII redaction. Use that when you want a small ASR step inside a pipeline. Use this notebook when you want a 153M stored student of Qwen3-ASR and a looped-transformer walkthrough.

**Compute vs storage.** 153M stored parameters does **not** mean 153M-worth of matmuls. Per-token compute stays close to the 1.7B teacher; the trimmed output table (37k vs 152k rows) is the only arithmetic saving. Merged weights need roughly teacher-sized accelerator memory while decoding.

**T4 / dtype.** Float16 on T4; the card's published numbers used bf16 autocast on datacenter GPUs. If a clip comes out empty, truncated, or badly drifted, reload in float32. Audio longer than 30 s is cut at quiet points; each chunk is capped at `min(448, 10 tokens/s + 24)` tokens.

**Tiny eval.** Five dummy LibriSpeech clips are a teaching smoke test. The card's 7.12% mean is over AMI, Earnings-22, GigaSpeech, LibriSpeech clean/other, SPGISpeech, and VoxPopuli, with checkpoints selected on those same test sets.

**Training-data terms.** SPGISpeech, GigaSpeech, and TED-LIUM 3 restrict how the *audio* may be used. That is separate from the weights licence.
